# generate_data

This script will be used to generate data to be loaded into our data base for testing the usage intelligence dashboard.

In [6]:
import random
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

SEED = 42
random.seed(SEED)

# --------Configuration---------
NUM_USERS = 180
DAYS = 60  
START_DATE = (datetime.now() - timedelta(days=DAYS)).date()

COMPANIES = ["Apex Analytics","Northstar Logistics","Summit Financial","Orion Digital","Vertex Solutions","Nimbus Technologies",
             "Atlas Supply Chain","BrightPath Healthcare","Horizon Energy","Redwood Ventures"]

DEPARTMENTS = ["Sales", "Engineering", "Procurement", "Marketing", "Finance", "IT", "HR", "Management"]
ROLES = ["Admin", "Manager", "Individual Contributor"]

# app cost (cost per month)
APP_SEAT_COST = {"Salesforce": 150,"Slack": 12,"Notion": 10,"Jira": 8,"HubSpot": 120,"Zendesk": 90,"Google Workspace": 12,}

# Department -> app affinity weights (higher = more likely)
DEPT_APP_WEIGHTS = {
    "Sales": {"Salesforce": 10, "HubSpot": 7, "Slack": 5, "Notion": 2, "Google Workspace": 2, "Jira": 1, "Zendesk": 1},
    "Engineering": {"Salesforce": 1, "HubSpot": 2, "Slack": 9, "Notion": 9, "Google Workspace": 9, "Jira": 9, "Zendesk": 1},
    "Procurement": {"Salesforce": 7, "HubSpot": 7, "Slack": 8, "Notion": 3, "Google Workspace": 2, "Jira": 6, "Zendesk": 9},
    "Marketing": {"Salesforce": 2, "HubSpot": 9, "Slack": 9, "Notion": 3, "Google Workspace": 8, "Jira": 1, "Zendesk": 1},
    "Finance": {"Salesforce": 4, "HubSpot": 7, "Slack": 4, "Notion": 4, "Google Workspace": 1, "Jira": 1, "Zendesk": 1},
    "IT": {"Salesforce": 2, "HubSpot": 3, "Slack": 7, "Notion": 4, "Google Workspace": 2, "Jira": 2, "Zendesk": 3},
    "HR": {"Salesforce": 1, "HubSpot": 3, "Slack": 6, "Notion": 1, "Google Workspace": 7, "Jira": 1, "Zendesk": 8},
    "Management": {"Salesforce": 8, "HubSpot": 4, "Slack": 5, "Notion": 4, "Google Workspace": 5, "Jira": 2, "Zendesk": 7}
}

def weighted_choice(weight_dict: dict) -> str:
    """
    This function randomly selects a key from a dictionary using the dictionary values as probability weights.    
    """
    items = list(weight_dict.items())
    apps, weights = zip(*items)
    return random.choices(apps, weights=weights, k=1)[0]

def random_datetime_within_days(days_back: int) -> datetime:
    """
    Generates a random datetime within a specified number of days in the past.
    """
    now = datetime.now()
    start = now - timedelta(days=days_back)
    # random second between start and now
    delta_seconds = int((now - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, delta_seconds))

def main():
    Path("../data").mkdir(parents=True, exist_ok=True)
    # ---------- Create users ----------
    users = []
    for i in range(1, NUM_USERS + 1):
        dept = random.choice(DEPARTMENTS)
        role = random.choices(ROLES, weights=[1, 2, 7], k=1)[0]  # mostly Individual Contributors
        company = random.choices(COMPANIES, weights=[4, 3, 3, 2, 2, 2, 2, 1, 1, 3], k=1)[0]
        created_at = random_datetime_within_days(365).strftime("%Y-%m-%d %H:%M:%S")

        users.append({
            "user_id": i,
            "company": company,
            "department": dept,
            "role": role,
            "created_at": created_at
        })

    users_df = pd.DataFrame(users)

    # Tags some users as power users and some as inactive
    user_ids = users_df["user_id"].tolist()
    power_users = set(random.sample(user_ids, k=max(1, int(0.12 * NUM_USERS))))   
    inactive_users = set(random.sample([u for u in user_ids if u not in power_users],
                                       k=max(1, int(0.10 * NUM_USERS))))        

    # ---------- Create usage_daily ----------
    usage_rows = []
    for day_offset in range(DAYS):
        d = (START_DATE + timedelta(days=day_offset)).strftime("%Y-%m-%d")

        for _, u in users_df.iterrows():
            uid = int(u["user_id"])
            dept = u["department"]

            # base probability this user is active on a given day
            # (power users active more often; inactive less often)
            if uid in inactive_users:
                active_prob = 0.10
            elif uid in power_users:
                active_prob = 0.75
            else:
                active_prob = 0.40

            # some weekdays/weekends effect
            weekday = (START_DATE + timedelta(days=day_offset)).weekday()
            if weekday >= 5:
                active_prob *= 0.75

            if random.random() > active_prob:
                continue  # no usage that day

            # chooses how many apps they used today (1-3 typically)
            apps_today = set()
            num_apps = random.choices([1, 2, 3], weights=[65, 25, 10], k=1)[0]
            while len(apps_today) < num_apps:
                apps_today.add(weighted_choice(DEPT_APP_WEIGHTS[dept]))

            for app in apps_today:
                # sessions / minutes / actions depend on power user vs normal
                if uid in power_users:
                    sessions = max(1, int(random.gauss(6, 2)))
                    minutes = max(5, int(random.gauss(55, 20)))
                    actions = max(5, int(random.gauss(35, 15)))
                    error_prob = 0.06
                else:
                    sessions = max(1, int(random.gauss(3, 1.5)))
                    minutes = max(3, int(random.gauss(25, 12)))
                    actions = max(1, int(random.gauss(12, 8)))
                    error_prob = 0.04

                errors = 0
                if random.random() < error_prob:
                    errors = random.choices([1, 2, 3], weights=[70, 20, 10], k=1)[0]

                usage_rows.append({
                    "date": d,
                    "user_id": uid,
                    "app": app,
                    "sessions": int(sessions),
                    "minutes_used": int(minutes),
                    "feature_actions": int(actions),
                    "errors": int(errors),
                    "seat_cost_usd": int(APP_SEAT_COST[app]),
                })

    usage_df = pd.DataFrame(usage_rows)
    usage_df[["sessions","minutes_used","feature_actions","errors","seat_cost_usd","user_id"]] = \
    usage_df[["sessions","minutes_used","feature_actions","errors","seat_cost_usd","user_id"]].astype(int)

    # de-duplicate
    usage_df = usage_df.drop_duplicates(subset=["date", "user_id", "app"])

    # ---------- Write CSVs ----------
    users_df.to_csv("../data/users.csv", index=False)
    usage_df.to_csv("../data/usage_daily.csv", index=False)
    
    print("Generated:")
    print(" - data/users.csv  (rows:", len(users_df), ")")
    print(" - data/usage_daily.csv (rows:", len(usage_df), ")")
    print("\nExample users:")
    print(users_df.head(3))
    print("\nExample usage:")
    print(usage_df.head(5))


if __name__ == "__main__":
    main()

Generated:
 - data/users.csv  (rows: 180 )
 - data/usage_daily.csv (rows: 5930 )

Example users:
   user_id              company   department                    role  \
0        1  Northstar Logistics  Engineering                   Admin   
1        2  Nimbus Technologies  Procurement  Individual Contributor   
2        3       Apex Analytics  Engineering  Individual Contributor   

            created_at  
0  2025-05-29 13:56:52  
1  2026-02-13 09:11:21  
2  2025-04-09 06:46:33  

Example usage:
         date  user_id               app  sessions  minutes_used  \
0  2026-01-02        1            Notion         3            24   
1  2026-01-02        2             Slack         1             3   
2  2026-01-02        3  Google Workspace         2             3   
3  2026-01-02        4           HubSpot         7            76   
4  2026-01-02        6  Google Workspace         1             3   

   feature_actions  errors  seat_cost_usd  
0               12       0             10  
1